In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Abarna Studio\AppData\Local\Temp\ipykernel_20656\4048863546.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print("Gemini API Loaded")

Gemini API Loaded


## Load Embedding Model

In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6311.09it/s]


In [4]:
db = FAISS.load_local(
    "../vectorstore/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS Loaded Successfully")

FAISS Loaded Successfully


In [5]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

## Load Gemini

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

In [7]:
prompt = ChatPromptTemplate.from_template(
"""
You are a helpful customer support assistant.

Answer ONLY using the context below.

If the answer is not present in the context, reply:

"I couldn't find that information in the knowledge base."

Context:
{context}

Question:
{question}
"""
)

In [8]:
parser = StrOutputParser()

In [9]:
def ask_chatbot(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    chain = prompt | llm | parser

    response = chain.invoke({

        "context": context,

        "question": question

    })

    return response

In [10]:
print(
    ask_chatbot(
        "How do I return a product?"
    )
)

f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Initiate a return from your order history within the eligible return window.


In [11]:
questions = [

    "How do I reset my password?",

    "How can I cancel my order?",

    "When will I receive my refund?",

    "How do I claim warranty?",

    "Do you accept UPI?",

    "What is your return policy?",

    "How can I contact customer support?"
]

In [12]:
for q in questions:
    print("="*80)
    print("Question:", q)
    print()
    print(ask_chatbot(q))
    print()

Question: How do I reset my password?



f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Use the 'Forgot Password' option on the login page.

Question: How can I cancel my order?



f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I couldn't find that information in the knowledge base.

Question: When will I receive my refund?



...

Yes, UPI payments are supported.

Question: What is your return policy?



f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Products may be returned within 30 days of delivery. Items should be in original condition and include all accessories. Damaged, defective, or incorrect items are eligible for free return pickup. Refunds are processed after inspection.

Question: How can I contact customer support?



f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I couldn't find that information in the knowledge base.



In [13]:
while True:
    q = input("\nYou: ")
    if q.lower() == "exit":
        break
    print()
    print("Bot:")
    print(ask_chatbot(q))


Bot:


f:\Guvi\Old\Projects\project7\venv_new\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Use the 'Forgot Password' option on the login page.
